In [2]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from collections import Counter
import os
import itertools # Para generar las 16 combinaciones
from scipy.spatial import KDTree

# --- FUNCIONES DE APOYO ---

def rgb_to_hex(color):
    return "#{:02x}{:02x}{:02x}".format(int(color[0]), int(color[1]), int(color[2]))

def hex_to_rgb(hex_code):
    hex_code = hex_code.lstrip('#')
    return tuple(int(hex_code[i:i+2], 16) for i in (0, 2, 4))

def get_text_color(rgb):
    luminance = 0.299 * rgb[0] + 0.587 * rgb[1] + 0.114 * rgb[2]
    return (255, 255, 255) if luminance < 128 else (0, 0, 0)

def get_exact_color_counts(image_path, top_n=20):
    img = cv2.imread(image_path)
    if img is None: return None
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    pixels = img.reshape(-1, 3)
    counts = Counter([tuple(p) for p in pixels])
    total_pixels = len(pixels)
    sorted_counts = counts.most_common(top_n)
    
    data_list = []
    for i, (color, count) in enumerate(sorted_counts):
        data_list.append({'ID': i, 'HEX': rgb_to_hex(color), 'Porcentaje': (count/total_pixels)*100, 'RGB': color})
    return pd.DataFrame(data_list)

# --- PROCESAMIENTO ---

def process_batch_automated(or_dir, df_colores, similar_colors, replace_dict, folder_name):
    # Creamos la carpeta con el nombre cat_0011 etc.
    if not os.path.exists(folder_name):
        os.makedirs(folder_name)

    palette_rgb = np.array(df_colores['RGB'].tolist())
    tree = KDTree(palette_rgb)

    mapping_rgb = {}
    for group_idx, target_hex in replace_dict.items():
        new_rgb = hex_to_rgb(target_hex)
        for color_id in similar_colors[group_idx]:
            row = df_colores.loc[df_colores['ID'] == color_id]
            if not row.empty:
                mapping_rgb[tuple(row['RGB'].values[0])] = new_rgb

    for img_name in os.listdir(or_dir):
        if not img_name.lower().endswith(('.png', '.jpg', '.jpeg')): continue
        
        img = cv2.imread(os.path.join(or_dir, img_name), cv2.IMREAD_UNCHANGED)
        if img is None: continue

        bgr = img[:, :, :3]
        alpha = img[:, :, 3] if img.shape[2] == 4 else None
        rgb_img = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        
        h, w, c = rgb_img.shape
        pixels_flat = rgb_img.reshape(-1, 3)
        _, indices = tree.query(pixels_flat)
        rgb_img_quantized = palette_rgb[indices].reshape(h, w, c)
        
        new_rgb_img = rgb_img_quantized.copy()
        for orig_rgb, target_rgb in mapping_rgb.items():
            mask = np.all(rgb_img_quantized == orig_rgb, axis=-1)
            new_rgb_img[mask] = target_rgb

        final_bgr = cv2.cvtColor(new_rgb_img, cv2.COLOR_RGB2BGR)
        final_img = cv2.merge([final_bgr, alpha]) if alpha is not None else final_bgr

        cv2.imwrite(os.path.join(folder_name, img_name), final_img)

# --- CONFIGURACIÓN DE COLORES (2 OPCIONES POR GRUPO) ---

# Editá estos colores a tu gusto (HEX)
piel_opts    = ["#D35400", "#7F8C8D"] # Naranja, Gris
manchas_opts = ["#FDFEFE", "#4E342E"] # Blanco, Marrón
ojos_opts    = ["#2ECC71", "#3498DB"] # Verde, Azul
orejas_opts  = ["#F1948A", "#D5DBDB"] # Rosado, Plateado

# Mapeo de IDs de tu imagen base (según lo que definimos antes)
similar_colors = [
    [1],                            # Grupo 0 -> Piel
    [2, 6, 9, 10, 11, 12, 15, 18],  # Grupo 1 -> Bordes (Usaremos un color fijo)
    [3, 7, 13, 14, 16, 17, 19],     # Grupo 2 -> Manchas
    [4],                            # Grupo 3 -> Ojos
    [5]                             # Grupo 4 -> Orejas
]

# --- EJECUCIÓN PRINCIPAL ---

df_colores = get_exact_color_counts('./cat_base/2a.png', top_n=20)

# Generamos las 16 combinaciones (0,0,0,0 hasta 1,1,1,1)
options = [0, 1]
combinations = list(itertools.product(options, repeat=4))

print(f"[*] Iniciando generación de {len(combinations)} variantes...")

for combo in combinations:
    # combo es una tupla como (0, 1, 0, 1)
    p, m, o, e = combo
    folder_id = f"{p}{m}{o}{e}"
    folder_name = f"cat_{folder_id}"
    
    # Construimos el diccionario de reemplazo
    replace_dict = {
        0: piel_opts[p],
        1: "#2C3E50",      # Bordes: Color oscuro fijo para que siempre se vea bien
        2: manchas_opts[m],
        3: ojos_opts[o],
        4: orejas_opts[e]
    }
    
    process_batch_automated("./cat_base/", df_colores, similar_colors, replace_dict, folder_name)
    print(f"[+] Generada: {folder_name}")

print("\n[!] Proceso finalizado. Revisá las carpetas generadas.")

[*] Iniciando generación de 16 variantes...
[+] Generada: cat_0000
[+] Generada: cat_0001
[+] Generada: cat_0010
[+] Generada: cat_0011
[+] Generada: cat_0100
[+] Generada: cat_0101
[+] Generada: cat_0110
[+] Generada: cat_0111
[+] Generada: cat_1000
[+] Generada: cat_1001
[+] Generada: cat_1010
[+] Generada: cat_1011
[+] Generada: cat_1100
[+] Generada: cat_1101
[+] Generada: cat_1110
[+] Generada: cat_1111

[!] Proceso finalizado. Revisá las carpetas generadas.
